<a href="https://colab.research.google.com/github/AyalaCohen-1/Projet_maths_info_Ayala_et_Aviva/blob/main/Projet%20maths%20info.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Projet Maths Info- Aviva Tordjman et Ayala Cohen

In [ ]:
#on importe les bibliotheques necessaires pour le projet
import pandas as pd
import numpy as np

####Partie 1: Compréhension des arbres de décision

Avant d'étudier et d'implementer notre modele il est nécessaire d'acquerir une comprehension approfondie des arbres de décision a l'aide de cette source: https://fr.wikipedia.org/wiki/Arbre_de_d%C3%A9cision_(apprentissage) .


####Partie 2 : Préparer les données

In [ ]:
nom_fichier = 'Heart_disease_cleveland_new.csv'
df = pd.read_csv(nom_fichier)
# Si le fichier contient des '?', on les remplace par NaN et on nettoie
df = df.replace('?', np.nan)
df = df.apply(pd.to_numeric, errors='coerce')
df = df.fillna(df.mean())

if 'condition' in df.columns:
    df = df.rename(columns={'condition': 'maladie'})

print(f"Fichier {nom_fichier} chargé avec succès !")
print(f"Nombre de lignes : {len(df)}")
df.head()

In [ ]:
#on identifie la dernière colonne (la cible)
nom_cible = df.columns[-1]
print(f"La colonne cible est : {nom_cible}")

#soit X (toutes les colonnes sauf la dernière) et y (la dernière colonne)
X = df.drop(nom_cible, axis=1).values
y = df[nom_cible].values

np.random.seed(42)
indices = np.arange(len(X))
np.random.shuffle(indices)

#On calcule la limite pour 80% (Entraînement) et 20% (Test)
limite = int(0.8 * len(X))

X_train = X[indices[:limite]]
y_train = y[indices[:limite]]
X_test = X[indices[limite:]]
y_test = y[indices[limite:]]

print(f"Données d'entraînement : {len(X_train)} lignes")
print(f"Données de test : {len(X_test)} lignes")

####Partie 3: Construction de l'arbre

In [ ]:
#on commence par creer la structure des arbres
class Node:
    def __init__(
        self,
        feature=None,
        threshold=None,
        left=None,
        right=None,
        value=None
    ):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

a) Pour cette partie on a deux criteres de division possibles : Gini impurity et l'entropie. En observant leurs formules respectives on remarque que celle de l'entropie est bien plus complexe que celle de la Gini Impurity;  Gini utilise une simple mise au carré ($p^2$), tandis que l'entropie nécessite le calcul d'un logarithme base 2 ($\log_{2}(p)$). Le calcul d'un logarithme est bien plus long que celui d'un carré.
Aussi, peu importe le critere choisi les résultats seront presque les memes j'ai donc choisi celui qui est le plus rapide a calculer.

In [ ]:
#partie a
#je choisis le critere de division Gini Impurity car il me semble plus simple que l'entropie

def gini(y):
    if len(y) == 0:
        return 0 #pour pas avoir de probleme de division par 0

    classes, counts = np.unique(y, return_counts=True)
    impurity = 1
    for count in counts:
        p = count / len(y)
        impurity -= p ** 2
    return impurity

b) Ici encore on a un choix a faire en deux algorithmes a implémenter : CART ou ID3. Le choix est assez simple sachant qu'on a choisi d'implémenter la Gini impurity l'algorithme associé est CART. Cet algorithme va créer l'arbre de décision en choisissant récursivement les divisions qui minimisent la Gini impurity pour chaque noeud. On aura donc a la fin un algorithme récursif de construction de l'arbre ou les noeuds auront pour enfants la meilleure division de branches pour avoir le diagnostic le plus exact possible.

c) On a choisi ici comme critere d'arret la profondeur de l'arbre. En la limitant a 10 on aura un résultat assez exact sans pour autant avoir des calculs interminables. Sans conditions d'arret le modele pourrait potentiellement se poursuivre presque a l'infini et on aura une feuille par cas de l'étude.

In [ ]:
#partie b
#on utilise l'algorithme CART car c'est celui associé a la Gini impurity
#cet algorithme va tester les differentes valeurs de nos données pour trouver celles ou la coupure entre sain et malade est la plus nette (qu'on aura d'un cote clairement un plus grand pourcentage de malades que de l'autre coté)
#il va ainsi construire un arbre de decision de maniere recursive en testant les differentes valeurs et choisissant les meilleures

class DecisionTree: #on cree une nouvelle classe pour l'arbre de decision c'est ce que l'algorithme va créer
#on implemente ici l'algorithme de cart pour que l'arbre soit créé selon celui ci
    def __init__(self, max_depth=10):#max depth servira pour le critere d'arret qu'on a choisi ici comme étant la profndeur maximale de l'arbre (on ne veut pas un arbre qui s'étende a l'infini)
        self.max_depth = max_depth
        self.root = None

    def fit(self, X, y):
        #on initialise la racine et lance la construction récursive de l'arbre
        self.root = self._build_tree(X, y, depth=0)

    def _build_tree(self, X, y, depth):
# partie c
        #criteres d'arret mis ici pour le bon fonctionnement de la creation de notre arbre recursif
        # Si on atteint la profondeur max qu'on a fixé (ici 10) ou si le noeud n'a qu'1 seule classe on arrete de chercher, ca nous empeche d'avoir un algo infini
        if depth >= self.max_depth or len(np.unique(y)) <= 1:
            classes, counts = np.unique(y, return_counts=True)
            return Node(value=classes[np.argmax(counts)]) # renvoie la classe majoritaire

        #
        best_feat, best_thresh, best_gini = None, None, float("inf")

        # On teste toutes les colonnes et tous les seuils possibles
        for feat in range(X.shape[1]):
            for thresh in np.unique(X[:, feat]):
                # On sépare les indices selon la condition
                left_idx = np.where(X[:, feat] <= thresh)[0]
                right_idx = np.where(X[:, feat] > thresh)[0]

                # Si la division ne met pas tout d'un seul cote
                if len(left_idx) > 0 and len(right_idx) > 0:
                    # Calcul du Gini pondéré des deux côtés
                    gini_val = (len(left_idx) * gini(y[left_idx]) + len(right_idx) * gini(y[right_idx])) / len(y)

                    # On sauvegarde si c'est le meilleur score
                    if gini_val < best_gini:
                        best_gini, best_feat, best_thresh = gini_val, feat, thresh

        # si aucune division n'ameliore le modele, on cree une feuille
        if best_feat is None:
            classes, counts = np.unique(y, return_counts=True)
            return Node(value=classes[np.argmax(counts)])

        left_idx = np.where(X[:, best_feat] <= best_thresh)[0]
        right_idx = np.where(X[:, best_feat] > best_thresh)[0]

        # le noeud s'appelle lui-même pour créer les branches suivantes et recommencer le processus de plus haut, c'est ca appliqué a chaque noeud créé de l'arbre qui va finir par créer l'arbre en entier
        return Node(
            feature=best_feat,
            threshold=best_thresh,
            left=self._build_tree(X[left_idx], y[left_idx], depth + 1),
            right=self._build_tree(X[right_idx], y[right_idx], depth + 1)
        )

    def predict(self, X):
        #Prédit la classe pour de nouvelles données
        # fonction interne pour descendre une ligne de données dans l'arbre
        def _traverse(x, node):
            if node.value is not None:
                return node.value # On a atteint une feuille, on donne le diagnostic associé a cette feuille (ya que 2 diagnostics possibles sain ou malade)

            # On descend à gauche ou à droite selon la condition
            if x[node.feature] <= node.threshold:
                return _traverse(x, node.left)
            return _traverse(x, node.right)

        # Applique la traversée à chaque ligne de X
        return np.array([_traverse(x, self.root) for x in X])

d) Le but de cette partie est de créer une fonction d'élagage qui va s'ajouter a la structure de notre arbre de décision. Cette fonction servira a réajuster l'arbre pour éviter les feuilles inutiles. On considere comme inutiles les feuilles enfants du meme noeud qui portent le meme diagnostic, on les supprime et associe le diagnostic au noeud parent. Cette fonction parcourt de maniere récursive l'arbre pour supprimer tout ces noeuds inutiles.

In [ ]:
#partie d
#on va écrire ici une fonction d'élagage pour optimiser l'arbre
#c'est une sorte de systeme pour que le systeme se réajuste par lui meme pour ne pas avoir de branches inutiles
def prune(self):
    if self.root is not None:
        self.prune_node(self.root)

def prune_node(self, node):
    if node.value is not None:#si on tombe sur une feuille on s'arrete et retourne la valeur du noeud contenant le diagnostic
        return

    if node.left is not None: # si on tombe sur une branche on continue d'elaguer en appelant recursivement la fonction a gauche et a droite
        self.prune_node(node.left)
    if node.right is not None:
        self.prune_node(node.right)

    if (node.left is not None and node.left.value is not None) and \
     (node.right is not None and node.right.value is not None):

        if node.left.value == node.right.value:#si deux feuilles ont la meme "conclusion" (malade ou sain) ca sert a rien d'en avoir 2 on enleve ces deux feuilles inutiles et donne le diagnostique au noeud pere des 2 feuilles inutiles
            node.value = node.left.value
            node.left = None
            node.right = None
            node.feature = None
            node.threshold = None

#on ajoute les deux dernieres fonctions a la structure de l'arbre pour qu'il puisse s'elager
DecisionTree.prune = prune
DecisionTree.prune_node = prune_node

####Partie 4: Evaluation de la performance

In [ ]:
# On calcule les 4 éléments de la matrice de confusion
def calculer_performance(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1)) # Vrais Positifs
    tn = np.sum((y_true == 0) & (y_pred == 0)) # Vrais Négatifs
    fp = np.sum((y_true == 0) & (y_pred == 1)) # Faux Positifs
    fn = np.sum((y_true == 1) & (y_pred == 0)) # Faux Négatifs

    # Calcul des mesures mathematiques (en faisant attention a la division par zéro)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    rappel = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * rappel) / (precision + rappel) if (precision + rappel) > 0 else 0

    return {
        "TP": tp, "TN": tn, "FP": fp, "FN": fn,
        "Précision": round(precision, 3),
        "Rappel": round(rappel, 3),
        "F1-Score": round(f1, 3)
    }

def afficher_matrice(perf):
    print("MATRICE DE CONFUSION")
    print(f"Sains bien classés (TN) : {perf['TN']} | Faux Malades (FP) : {perf['FP']}")
    print(f"Faux Sains (FN) : {perf['FN']}      | Malades bien classés (TP) : {perf['TP']}")